In [1]:
pip install scipy

You should consider upgrading via the '/Users/rochanvanam/Documents/python_projects/quantproject/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import math
from scipy.stats import norm

In [3]:
import numpy as np

def american_option_binomial(S, K, T, r, sigma, n, option_type="call"):
    dt = T / n  # Time step
    u = np.exp(sigma * np.sqrt(dt))  # Up factor
    d = 1 / u  # Down factor
    p = (np.exp(r * dt) - d) / (u - d)  # Risk-neutral probability
    
    # Initialize asset prices at maturity
    stock_price = np.zeros(n + 1)
    option_price = np.zeros(n + 1)
    
    for j in range(n + 1):
        stock_price[j] = S * (u ** (n - j)) * (d ** j)
    
    # Compute option value at maturity
    if option_type == "call":
        option_price = np.maximum(stock_price - K, 0)
    else:
        option_price = np.maximum(K - stock_price, 0)
    
    # Step backward through the tree
    for i in range(n - 1, -1, -1):
        for j in range(i + 1):
            stock_price[j] = S * (u ** (i - j)) * (d ** j)
            option_value = (p * option_price[j] + (1 - p) * option_price[j + 1]) * np.exp(-r * dt)
            
            if option_type == "call":
                option_price[j] = max(stock_price[j] - K, option_value)
            else:
                option_price[j] = max(K - stock_price[j], option_value)
    
    return option_price[0]

# Example usage:
S = 100  # Current stock price
K = 100  # Strike price
T = 1    # Time to maturity in years
r = 0.04322 # Risk-free rate, 10 year american bonds rate
sigma = 0.2 # Volatility
n = 100  # Number of time steps

american_call_price = american_option_binomial(S, K, T, r, sigma, n, "call")
american_put_price = american_option_binomial(S, K, T, r, sigma, n, "put")

print(f"American Call Option Price: {american_call_price:.4f}")
print(f"American Put Option Price: {american_put_price:.4f}")

American Call Option Price: 10.0729
American Put Option Price: 6.2920


In [4]:
import scipy.stats as si

def american_option_baw(S, K, T, r, sigma, option_type="call"):
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    N_d1 = si.norm.cdf(d1)
    N_d2 = si.norm.cdf(d2)
    
    # Black-Scholes European option price
    if option_type == "call":
        european_price = S * N_d1 - K * np.exp(-r * T) * N_d2
    else:
        european_price = K * np.exp(-r * T) * si.norm.cdf(-d2) - S * si.norm.cdf(-d1)
    
    # Approximate early exercise premium using BAW
    q = 2 * r / (sigma ** 2)
    beta = (1 - np.exp(-r * T)) / (1 - np.exp(-q * T))
    
    if option_type == "call":
        S_star = K / (1 - beta)
        if S >= S_star:
            return S - K  # Immediate exercise
        else:
            return european_price + (S_star - S) * beta
    else:
        S_star = K / (1 + beta)
        if S <= S_star:
            return K - S  # Immediate exercise
        else:
            return european_price + (S - S_star) * beta

# Example usage:
S = 100  # Current stock price
K = 100  # Strike price
T = 1    # Time to maturity in years
r = 0.04322 # Risk-free rate
sigma = 0.2 # Volatility

american_call_price_baw = american_option_baw(S, K, T, r, sigma, "call")
american_put_price_baw = american_option_baw(S, K, T, r, sigma, "put")

print(f"American Call Option Price (BAW): {american_call_price_baw:.4f}")
print(f"American Put Option Price (BAW): {american_put_price_baw:.4f}")

American Call Option Price (BAW): 10.3328
American Put Option Price (BAW): 6.0810
